In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 8.2 Institutional Pattern Discovery II: Unsupervised Learning with Survey Data
- K-Means clustering on the Module 7 master matrix
- Elbow method to pick K
- Cluster profiling with raw values + student-voice validation

## Setup

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pd.options.display.max_columns = None
np.random.seed(15)
random.seed(15)

df_ml_train = pd.read_csv('../data/ML_SURVEY_MASTER_TRAIN.csv')
X_train = df_ml_train.drop(columns=['SEM_3_STATUS'])
print("Master matrix:", df_ml_train.shape)

## Reconstruct Raw Values for Interpretability
Cluster on the scaled/encoded matrix, but profile using raw (unscaled) values so cluster summaries are human-readable.

In [ ]:
df_training = pd.read_csv('../data/training.csv')
df_training1 = df_training.drop(columns=['SEM_3_STATUS'])
X_raw = pd.concat([df_training1, X_train.iloc[:, 19:]], axis=1)
print("Raw reference frame:", X_raw.shape)

## Choose K (Elbow Method)

In [ ]:
import plotly.express as px

inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertia.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertia, markers=True,
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'},
              title='Elbow Method — Choosing K for K-Means')
fig.show()

## Fit K-Means & Profile Clusters

In [ ]:
OPTIMAL_K = 3  # update based on the elbow plot above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans.fit(X_train)
X_raw['Cluster'] = kmeans.labels_

print("Cluster sizes:")
print(X_raw['Cluster'].value_counts().sort_index())

profile_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
                'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
profile_cols = [c for c in profile_cols if c in X_raw.columns]
cluster_profile = X_raw.groupby('Cluster')[profile_cols].mean().round(3)
print("\nCluster Numeric Profile:")
cluster_profile

## Visualize with PCA

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(X_train)

df_plot = pd.DataFrame({'PC1': coords[:, 0], 'PC2': coords[:, 1],
                        'Cluster': X_raw['Cluster'].astype(str)}, index=X_raw.index)

fig = px.scatter(df_plot, x='PC1', y='PC2', color='Cluster',
                 title=f'K-Means Clusters (K={OPTIMAL_K}) — PCA Projection')
fig.show()

## Hear the Student Voice by Cluster

In [ ]:
ML_Survey_Data = pd.read_csv('../data/ML_Survey_Data.csv')

print("Representative Comments by Cluster:")
for c in sorted(X_raw['Cluster'].unique()):
    index = X_raw[X_raw['Cluster'] == c].index
    subset = ML_Survey_Data.iloc[index, :]
    examples = subset.sample(min(2, len(subset)), random_state=42)['Free_Response_Text'].tolist()
    print(f"\nCluster {c} (n={len(subset)}):")
    for ex in examples:
        print(f"  - {ex}")

## Save the Cluster Model

In [ ]:
import pickle, os
os.makedirs('../models/', exist_ok=True)
with open('../models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print("KMeans model saved to ../models/kmeans_model.pkl")

## Summary
- Unsupervised — no target variable, we discover structure instead of predicting it.
- Elbow method picks K, cluster profiling (on raw values) makes clusters interpretable for advisors.
- Pairing numeric profiles with real student comments gives the clusters a human voice, not just statistics.

**Next:** 8.3 uses this same master matrix for supervised prediction.